In [1]:
import os
import json
import numpy as np
import json
import math
from collections import defaultdict
import matplotlib.pyplot as plt

In [2]:
def number_conversion(num_str):
    """
    Convertit les valeurs numériques contenues dans les json des documents comptables (stockées
    sous format str) en float.
    """
    if not num_str:
        return None
    
    clean_str = str(num_str).strip()
    is_negative = False
    is_percentage = False
    
    # Gestion des nombres négatifs où le - est à la fin (ex : 156,00-)
    if clean_str.endswith('-'):
        is_negative = True
        clean_str = clean_str[:-1]

    # Gestion des pourcentages
    if clean_str.endswith('%'):
        is_percentage = True
        clean_str = clean_str[:-1]
        
    # Suppression des séparateurs de milliers (.) et remplacement de la décimale (,)
    clean_str = clean_str.replace('.', '').replace(',', '.').replace(' ', '')
    
    try:
        val = float(clean_str)
        if is_negative :
            val = -val
        if is_percentage :
            val = val / 100
        return val
    except ValueError:
        return None
    

def verification_valeurs_numeriques(chemin_fichier):


    # Récupération du fichier json désigné par le chemin indiqué en argument
    if not os.path.exists(chemin_fichier):
        print(f"❌ Erreur : Le fichier '{chemin_fichier}' est introuvable.")
        return

    try:
        with open(chemin_fichier, 'r', encoding='utf-8') as f:
            doc_json = json.load(f)

        print(f"--- Analyse de la facture : {doc_json.get('titre', 'Inconnue')} ---")
        
        contenu = doc_json.get('contenu', [])
        totaux = doc_json.get('totaux', {})
        
        # Extraction des totaux globaux du document
        total_ht_declare = number_conversion(totaux.get('montant_total_ht'))
        total_tva_declare = number_conversion(totaux.get('montant_tva'))
        total_ttc_declare = number_conversion(totaux.get('a_payer'))
        
        # Stockage des messages d'erreurs et de toutes les valeurs numériques (pour comparer avec la loi de Benford)
        errors = []
        all_numbers = []
        somme_lignes_calculee = 0.0
        
        # Vérification ligne par ligne
        print("\n[Détail Lignes]")
        for i, item in enumerate(contenu):
            # Récupération des différentes valeurs numériques relatives à l'article i
            qte = number_conversion(item.get('quantite'))
            poids = number_conversion(item.get('poids'))
            pu = number_conversion(item.get('pu'))
            decote = number_conversion(item.get('decote'))
            montant_unitaire_ht = number_conversion(item.get('prix_ht'))
            montant_ligne = number_conversion(item.get('montant'))
            
            # Ajout du montant de l'article à la somme globale
            somme_lignes_calculee += montant_ligne
            
            # Ajout aux nombres pour Benford (on prend tous les chiffres significatifs trouvés)
            all_numbers.extend([x for x in [qte, poids, pu, decote, montant_unitaire_ht, montant_ligne] if x is not None])
            # Vérification mathématique de la ligne
            # On vérifie que le montant unitaire hors taxe est bien égal à (prix unitaire au kilo - décote)
            if decote :
                if decote < 1 :
                    montant_unitaire_calcule = pu * (1 - decote)
                else :
                    montant_unitaire_calcule = pu + decote

            else :
                montant_unitaire_calcule = pu

            if poids is None :
                if qte :
                    montant_calcule = qte * montant_unitaire_calcule
                else :
                    montant_calcule = None
            else :
                montant_calcule = (poids/1000) * montant_unitaire_calcule


            if montant_unitaire_ht is None :

                if montant_calcule :
            
                    # On laisse une marge d'erreur de 0.01 pour les arrondis flottants liés à notre calcul
                    if abs(montant_calcule - montant_ligne) > 0.01:
                        errors.append(f"⚠️ Ligne {i+1} ({item.get('libelle')}): Incohérence mathématique. "
                                    f"Poids({poids}) x PU({pu}) = {montant_calcule:.2f}, "
                                    f"mais montant affiché = {montant_ligne}")
                    else :
                        print(f"Ligne {i+1} : OK")

                else :
                    pass

            else :

                if montant_calcule :
                    # On vérifie que le montant unitaire hors taxe est bien égal à ce que l'on veut
                    # On laisse une marge d'erreur de 0.01 pour les arrondis flottants liés à notre calcul
                    if abs(montant_unitaire_calcule - montant_unitaire_ht) > 0.01:
                        errors.append(f"⚠️ Ligne {i+1} ({item.get('libelle')}) : Incohérence mathématique. "
                                    f"PU({pu}) - decote({decote} = {montant_unitaire_calcule:.2f}, "
                                    f"mais montant affiché = {montant_unitaire_ht}")
                    
                    # On vérifie que le montant de l'article est bien égal à (poids en kg) x montant unitaire hors taxe
                    if abs(montant_calcule - montant_ligne) > 0.01:
                        errors.append(f"⚠️ Ligne {i+1} ({item.get('libelle')}): Incohérence mathématique. "
                                    f"Poids({poids}) x PU({pu}) = {montant_calcule:.2f}, "
                                    f"mais montant affiché = {montant_ligne}")
                    else:
                        print(f"Ligne {i+1} : OK")

                else : 
                    pass

        print(f"\n[Vérification Sommes]")
        # On vérifie que la somme des montants des articles est bien égale au montant total hors taxe déclaré
        if abs(somme_lignes_calculee - total_ht_declare) < 0.01:
            print(f"✅ Somme des articles ({somme_lignes_calculee:.2f}) == Total HT déclaré.")
        else:
            print(f"❌ ERREUR : Somme des articles ({somme_lignes_calculee:.2f}) != Total HT ({total_ht_declare}).")

        # On vérifie que le montant total ttc est bien égal au montant total hors taxe déclaré + la tva déclarée
        # On pourra checker aussi que la tva est bien calculée
        calcul_ttc = total_ht_declare + total_tva_declare
        
        if abs(calcul_ttc - total_ttc_declare) < 0.01:
            print(f"✅ Cohérence Taxes : HT ({total_ht_declare}) + TVA ({total_tva_declare}) == A Payer ({total_ttc_declare})")
        else:
            print(f"❌ ERREUR Taxes : HT + TVA = {calcul_ttc:.2f}, mais 'A Payer' = {total_ttc_declare}")

        # On compare la distribution des valeurs numériques avec la loi de Benford
        # print(f"\n[Loi de Benford]")
        # On ne garde que les nombres > 0 pour Benford
        nombres_significatifs = [n for n in all_numbers if n > 0]
        premier_chiffre_count = defaultdict(int)
        
        # On récupère le premier chiffre significatif pour chaque valeur numérique
        for n in nombres_significatifs:
            first_digit = int(str(n).replace('.', '').lstrip('0')[0])
            premier_chiffre_count[first_digit] += 1
            
        total_obs = len(nombres_significatifs)
        # print(f"Analyse sur {total_obs} nombres extraits.")
        # print(f"{'Chiffre':<8} | {'Réel %':<10} | {'Benford %':<10} | {'Status'}")
        # print("-" * 45)

        observed_freqs = []
        expected_freqs = []
        
        for d in range(1, 10):
            observed_freq = (premier_chiffre_count[d] / total_obs) * 100
            expected_freq = math.log10(1 + 1/d) * 100

            observed_freqs.append(observed_freq)
            expected_freqs.append(expected_freq)
            
            # Si on observe pour un chiffre, un écart > 15 % en fréquence par rapport à la loi de Benford,
            # On peut estimer qu'il y a une sur (ou sous) représentation suspecte de ce chiffre dans notre set
            # diff = abs(observed_freq - expected_freq)
            # status = "⚠️" if diff > 15 and total_obs > 20 else " " 
            
            # print(f"{d:<8} | {observed_freq:5.1f}%     | {expected_freq:5.1f}%     | {status}")

        # Affichage des erreurs détectées
        if errors:
            print("\n⚠️ ATTENTION : Incohérences détectées dans les lignes :")
            for e in errors:
                print(f"  - {e}")
        else:
            print("\n✅ Aucune incohérence de ligne détectée.")

        plt.figure()
        plt.plot(observed_freqs,label="Fréquence observée")
        plt.plot(expected_freqs,label="Loi de Benford")
        plt.title("Comparaison de la fréquence de chaque chiffre avec la loi de Benford")
        plt.legend()


    except json.JSONDecodeError:
        print(f"❌ Erreur : Le fichier '{chemin_fichier}' n'est pas un JSON valide.")
    except Exception as e:
        print(f"❌ Une erreur inattendue est survenue : {e}")

In [7]:
test_json = "/home/oclaich/PIE/deepfake_detector/code_valentino/extracted/RB5.json"

verification_valeurs_numeriques(test_json)

--- Analyse de la facture : Inconnue ---

[Détail Lignes]

[Vérification Sommes]
❌ Une erreur inattendue est survenue : unsupported operand type(s) for -: 'float' and 'NoneType'
